# 股價情緒分析 Pipeline v2（修正版）

## 修正摘要

| Bug | 問題 | 修正方式 |
|-----|------|---------|
| **A** | LabelEncoder 污染 `y`（`[-1,1]`→`[0,1]`）導致 confusion matrix 錯亂 | 統一維持 `[-1, 1]`；`evaluate` 動態取 labels；移除 LabelEncoder 呼叫 |
| **B** | 日級合併 overfit（148 樣本 vs 400 特徵 = 2.7x） | 新增 `AGGREGATION_LEVEL` 超參數 (`'article'`/`'daily'`)，預設 article |
| **C** | Step 5 窗口型態混亂 | 新增 `WINDOW_TYPE` (`'calendar'`/`'trading'`)；加 `MIN_TRAIN_SAMPLES` 門檻 |
| **D** | 無 embargo，label leakage | 新增 `EMBARGO_DAYS=5`，train/test 間強制保留 buffer |
| **M1** | `START_DATE` 未定義 | 在 Step 0 定義，預設 `'2024-01-01'` |
| **M3** | 跨平台近重複未處理 | Step 1.5 加 MinHash LSH 去重（`datasketch`） |

## 新增功能

- **Article-level 訓練 + Daily Aggregate 預測**：`DAILY_PREDICTION=True` 時，article 訓練後測試時按日平均 proba → 日級判斷（兼得兩種好處）
- **sklearn Pipeline 綁緊**：vectorizer + selector + classifier 綁在一個 Pipeline，`fit` 時自動 per-fold，從架構上杜絕 leakage
- **診斷 cell**：每次訓練前印出 y 分布、特徵/樣本比、關鍵詞覆蓋率

## 你可以切換的超參數

```python
AGGREGATION_LEVEL = 'article'   # 'article' | 'daily'
DAILY_PREDICTION  = True         # article 訓練 + daily 預測
DEDUP_ENABLED     = True         # MinHash 去重開關
EMBARGO_DAYS      = 5
WINDOW_TYPE       = 'trading'   # 'calendar' | 'trading'
WINDOW_DAYS       = 30
```


---
# Step 0 — 總參數區與共用工具


In [1]:
# ═══════════════ 總參數區 ═══════════════

# ── 目標公司 ──
COMPANY_ID   = '2603'
COMPANY_NAME = '長榮'

# ── 資料時間範圍 ──
START_DATE = None           # None = 資料庫最早日期
END_DATE   = None           # None = 資料庫最新日期

# ── Step 1: 篩選關鍵字 ──
INCLUDE_KEYWORDS = ['長榮', '長榮海運', '2603', '海運', '貨櫃三雄']
EXCLUDE_KEYWORDS = ['長榮航空', '長榮空服', '長榮桂冠', '長榮大學',
                    '長榮中學', '長榮女中', '張榮發']

# ── Step 2: 斷詞 ──
TOKENIZE_METHOD = 'monpa'   # 'monpa' | 'ngram'
NGRAM_RANGE     = (1, 3)    # TF-IDF 詞級 N-gram 範圍
NGRAM_MIN       = 2         # 字元級 ngram（只在 TOKENIZE_METHOD='ngram' 時用）
NGRAM_MAX       = 4

# ── Step 1.5: 去重 ──
DEDUP_ENABLED   = False
DEDUP_THRESHOLD = 0.8
DEDUP_NUM_PERM  = 128

# ── Step 3: 貼標 ──
N_DAYS = 7      # D+n 交易日後漲跌
SIGMA  = 0.01   # 漲跌門檻（1%）

# ── Step 3: 向量化與特徵選取 ──
TOP_K_WORDS = 150   # chi2 保留特徵數

# ── Step 3: 類別平衡 ──
# 'class_weight' : 訓練時補償（推薦，不損失資料）
# 'undersample'  : 與朋友對齊，平衡漲跌天數
BALANCE_METHOD = 'class_weight'

# ── Step 3: 資料切分 ──
SPLIT_METHOD = 'random'   # 'random' | 'time'
TEST_RATIO   = 0.2
RANDOM_SEED  = 42
EMBARGO_DAYS = 0          # 時序切割緩衝天數（0=關閉，'time'模式才有效）

# ── Step 5: 移動回測 ──
WINDOW_DAYS      = 40     # 滑動窗口天數（calendar days）
MIN_ARTICLES     = 3      # 一天至少幾篇文章才出手
MIN_TRAIN_DAYS   = 20     # 訓練窗口至少幾天才訓練
VOTE_MARGIN      = 0.0    # 信心門檻（0=全出手，0.75=只在高信心時出手）

# ── 檔名 ──
PATH_ARTICLES_RAW       = 'articles_raw.csv'
PATH_ARTICLES_FILTERED  = 'articles_filtered_out.csv'
PATH_ARTICLES_DEDUPED   = 'articles_deduped.csv'
PATH_TOKENIZED_CACHE    = f'{COMPANY_NAME}_tokenized.pkl'   # 斷詞快取
PATH_ARTICLES_DAILY     = 'articles_daily.csv'              # daily 合併後
PATH_X_TRAIN            = 'X_train.npz'
PATH_X_TEST             = 'X_test.npz'
PATH_Y_TRAIN            = 'y_train.npy'
PATH_Y_TEST             = 'y_test.npy'
PATH_DATES_TRAIN        = 'dates_train.npy'
PATH_DATES_TEST         = 'dates_test.npy'
PATH_KEYWORDS_ANALYSIS  = 'keywords_analysis.csv'

print("✅ 超參數已設定")
print(f"  N_DAYS={N_DAYS}, SIGMA={SIGMA}")
print(f"  BALANCE_METHOD={BALANCE_METHOD}")
print(f"  SPLIT_METHOD={SPLIT_METHOD}, EMBARGO_DAYS={EMBARGO_DAYS}")
print(f"  WINDOW_DAYS={WINDOW_DAYS}, VOTE_MARGIN={VOTE_MARGIN}")
print(f"  DEDUP_ENABLED={DEDUP_ENABLED}")


✅ 超參數已設定
  N_DAYS=7, SIGMA=0.01
  BALANCE_METHOD=class_weight
  SPLIT_METHOD=random, EMBARGO_DAYS=0
  WINDOW_DAYS=40, VOTE_MARGIN=0.0
  DEDUP_ENABLED=False


In [2]:
# ── 資料庫連線設定 ──
import os
from dotenv import load_dotenv
load_dotenv()
DB_CONFIG = {
    'host':     os.getenv('DB_HOST',     'localhost'),
    'user':     os.getenv('DB_USER',     'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME',     'bda2026'),
    'charset':  os.getenv('DB_CHARSET',  'utf8mb4'),
}


In [9]:
# ── 套件安裝（首次執行才需要）──
# %pip install stopwordsiso TCSP opencc xgboost lightgbm 
%pip install datasketch


Note: you may need to restart the kernel to use updated packages.


In [3]:
# ── 共用工具 ──
import re, bisect, csv, pickle
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import pymysql
from scipy import sparse


def db_connect():
    return pymysql.connect(**DB_CONFIG)


def is_valid_token(token: str) -> bool:
    """過濾 URL、純英數、雜訊 token。中文字需佔 80% 以上。"""
    if not token or len(token) < 2:
        return False
    if re.search(r'https?://', token) or re.search(r'www\.', token):
        return False
    chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', token))
    if chinese_chars == 0:
        return False
    if chinese_chars / len(token) < 0.8:
        return False
    return True


# ── 四層停用詞（保留原本邏輯）──
print("正在載入停用詞...")
from stopwordsiso import stopwords as iso_stopwords
from TCSP import read_stopwords_list
import requests
from opencc import OpenCC

BASE_STOPWORDS = set(iso_stopwords("zh")) | set(read_stopwords_list())

cc = OpenCC('s2twp')
urls = {
    "baidu": "https://raw.githubusercontent.com/goto456/stopwords/master/baidu_stopwords.txt",
    "hit":   "https://raw.githubusercontent.com/goto456/stopwords/master/hit_stopwords.txt",
    "scu":   "https://raw.githubusercontent.com/goto456/stopwords/master/scu_stopwords.txt",
}
goto456_words = set()
for name, u in urls.items():
    try:
        r = requests.get(u, timeout=10); r.raise_for_status()
        for w in r.text.splitlines():
            w = w.strip()
            if w:
                goto456_words.add(cc.convert(w))
    except Exception as e:
        print(f"⚠️ 無法下載 {name}: {e}")

CUSTOM_STOPWORDS = {
    # PTT 稱謂
    '小弟','大大','鄉民','版友','各位','請問','樓主','推主',
    '大家','大家好','您好','謝謝','感謝','不客氣',
    # PTT 回文樣板
    '引述','看板','批踢踢','發信站','轉錄至','Re','Fw','※',
    # 新聞樣板
    '記者','報導','撰文','原文連結','來源','全文','摘要',
    '本報訊','綜合報導','資料來源','圖片來源','編輯',
    # 填充語
    '覺得','應該','好像','似乎','其實','這樣','然後',
    '就是','所以','但是','不過','雖然','因為','如果',
    '可以','可能','已經','一直','真的','還是','還有',
    '一下','一些','一樣','這個','那個','什麼','怎麼',
    # 口語雜訊
    '哈哈','哈哈哈','XD','xd','QQ','嗚嗚','欸','啊','喔','哦','呵','嗯',
    # HTML 殘渣
    '閱讀全文','繼續閱讀','相關新聞','延伸閱讀','廣告','更多內容','訂閱',
    # 時間詞
    '今天','昨天','明天','今日','昨日','明日',
    '上週','下週','本週','這週','上個月','下個月','今年','去年','明年',
    # 財經樣板動詞
    '表示','指出','認為','說明','顯示','指稱',
    '根據','依據','根本','目前','近期','最近',
    # 基本虛詞
    '的','了','是','在','有','和','與','或','但','也','都','就',
    '這','那','我','你','他','她','它','們',
    '我們','你們','他們',
    # 連接詞
    '以及','同時','此外','另外','然而','即使','儘管',
    '無論','不管','只要','只有','除了','除非',
    '假如','要是','既然','因此','於是','進而','從而',
    # 介詞
    '關於','對於','來自','由於','透過','通過','藉由','基於','針對',
    # 無鑑別力動詞
    '提出','提供','進行','包括','需要','帶來','造成','導致','預計',
    # 其他
    '一個','不是','很多','如何','非常','相關',
    '其中','其他','之前','之後','方面','時間',
}

STOPWORDS = BASE_STOPWORDS | goto456_words | CUSTOM_STOPWORDS
print(f"✅ 停用詞載入完成：{len(STOPWORDS)} 個")


def get_ngrams(text: str, min_n: int = None, max_n: int = None) -> list:
    """N-gram 字元切割：只保留中/英/數。"""
    min_n = min_n or NGRAM_MIN
    max_n = max_n or NGRAM_MAX
    clean_text = ''.join(re.findall(r'[\u4e00-\u9fffA-Za-z0-9]+', text))
    ngrams = []
    for n in range(min_n, max_n + 1):
        for i in range(len(clean_text) - n + 1):
            gram = clean_text[i:i + n]
            if re.search(r'[\u4e00-\u9fffA-Za-z]', gram):
                ngrams.append(gram)
    return ngrams


def tokenize(text: str, method: str = None) -> list:
    """依 TOKENIZE_METHOD 切割文字並過濾 stopwords。"""
    method = method or TOKENIZE_METHOD
    if method == 'monpa':
        import monpa
        from monpa import utils as monpa_utils
        tokens = []
        for sent in monpa_utils.short_sentence(text):
            for term in monpa.cut(sent):
                term = term.strip()
                if len(term) > 1:
                    tokens.append(term)
    elif method == 'ngram':
        tokens = get_ngrams(text)
    else:
        raise ValueError(f'未知的 TOKENIZE_METHOD: {method}')
    return [t for t in tokens if is_valid_token(t) and t not in STOPWORDS]


print('✅ 共用工具已載入')


正在載入停用詞...
✅ 停用詞載入完成：2785 個
✅ 共用工具已載入


---
# Step 1 — 篩選相關文章

從 `stock_text` 擷取公司相關文章、排除京華城弊案等雜訊、過濾排行榜與數字比例過高的文章。


In [4]:
# ── 取得股價資料時間範圍 ──
conn = db_connect()
cursor = conn.cursor()

cursor.execute(
    'SELECT MIN(trade_date), MAX(trade_date) FROM stock_prices WHERE company_id = %s',
    (COMPANY_ID,)
)
db_min_date, db_max_date = cursor.fetchone()

# ⭐ 使用 START_DATE / END_DATE 覆寫（M1 修正）
min_date = START_DATE if START_DATE else db_min_date
max_date = END_DATE   if END_DATE   else db_max_date

print(f'股價資料範圍：{min_date} ～ {max_date}')

# ── 動態組 SQL ──
search_keywords = INCLUDE_KEYWORDS + [COMPANY_ID]
include_clause = ' OR '.join(['(title LIKE %s OR content LIKE %s)'] * len(search_keywords))
include_params = [p for kw in search_keywords for p in (f'%{kw}%', f'%{kw}%')]

if EXCLUDE_KEYWORDS:
    exclude_clause = ' AND ' + ' AND '.join(
        ['(title NOT LIKE %s AND content NOT LIKE %s)'] * len(EXCLUDE_KEYWORDS)
    )
    exclude_params = [p for kw in EXCLUDE_KEYWORDS for p in (f'%{kw}%', f'%{kw}%')]
else:
    exclude_clause, exclude_params = '', []

sql_query = f"""
    SELECT no, post_time, title, content, s_name
    FROM stock_text
    WHERE ({include_clause})
      {exclude_clause}
      AND (content_type = 'main' OR content_type IS NULL)
      AND DATE(post_time) >= %s
      AND DATE(post_time) <= %s
    ORDER BY post_time
"""

print('正在從資料庫篩選...')
cursor.execute(sql_query, include_params + exclude_params + [min_date, max_date])
rows = cursor.fetchall()
print(f'關鍵字精確篩選後：{len(rows)} 篇')

cursor.close()
conn.close()


股價資料範圍：2023-03-01 ～ 2025-02-27
正在從資料庫篩選...
關鍵字精確篩選後：19382 篇


In [5]:
# ── Boilerplate 過濾 ──
BOILERPLATE_TITLES = ['買賣超排行', '投信買賣', '外資買賣', '主力買賣', '法人買賣']

def is_boilerplate(title: str, content: str) -> bool:
    text = (title or '') + (content or '')
    if any(p in (title or '') for p in BOILERPLATE_TITLES):
        return True
    if len(text) == 0:
        return True
    if sum(c.isdigit() for c in text) / len(text) > 0.3:
        return True
    if len(text) < 50:
        return True
    return False

rows_kept    = [r for r in rows if not is_boilerplate(r[2], r[3])]
rows_removed = [r for r in rows if     is_boilerplate(r[2], r[3])]

print(f'篩選前：{len(rows)} 篇')
print(f'過濾掉：{len(rows_removed)} 篇')
print(f'保留：  {len(rows_kept)} 篇')


篩選前：19382 篇
過濾掉：4811 篇
保留：  14571 篇


In [61]:
# ── 儲存 CSV ──
HEADER = ['no', 'post_time', 'title', 'content', 's_name']

for path, data, label in [
    (PATH_ARTICLES_RAW,      rows_kept,    '保留'),
    (PATH_ARTICLES_FILTERED, rows_removed, '過濾'),
]:
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(data)
    print(f'已儲存 {path}（{label}）：{len(data)} 筆')

for data, label in [(rows_kept, '保留'), (rows_removed, '過濾')]:
    print(f'\n===== {label}文章平台分布 =====')
    for src, cnt in Counter(r[4] for r in data).most_common():
        print(f'  {src}: {cnt} 篇')


已儲存 articles_raw.csv（保留）：8833 筆
已儲存 articles_filtered_out.csv（過濾）：2658 筆

===== 保留文章平台分布 =====
  Yahoo股市: 5655 篇
  Ptt: 1492 篇
  Yahoo新聞: 1073 篇
  Dcard: 582 篇
  Mobile01: 31 篇

===== 過濾文章平台分布 =====
  Yahoo股市: 1661 篇
  Ptt: 897 篇
  Yahoo新聞: 50 篇
  Dcard: 49 篇
  Mobile01: 1 篇


---
# Step 1.5 — ⭐ 近重複去重（MinHash LSH）

**M3 修正**：Yahoo 新聞常被 PTT/Dcard/Mobile01 轉貼，同一則原文在不同平台出現 2–3 次。
時序切割時，近似副本可能分到 train/test，造成偽相關。

使用 `datasketch` MinHash LSH，threshold=0.8 找出相似度 >80% 的 cluster，每個 cluster 只保留**最早發佈者**。


In [34]:
# ── MinHash 去重 ──
if DEDUP_ENABLED:
    from datasketch import MinHash, MinHashLSH

    def text_shingles(text, k=5):
        """切成 k-shingles（字元級）。"""
        text = re.sub(r'\s+', '', text or '')
        return {text[i:i+k] for i in range(len(text) - k + 1)}

    # 載入 raw articles
    df_raw = pd.read_csv(PATH_ARTICLES_RAW)
    df_raw['post_time'] = pd.to_datetime(df_raw['post_time'])
    df_raw = df_raw.sort_values('post_time').reset_index(drop=True)
    print(f'載入 {len(df_raw)} 篇進行去重...')

    # 建立 MinHash
    lsh = MinHashLSH(threshold=DEDUP_THRESHOLD, num_perm=DEDUP_NUM_PERM)
    mh_store = {}
    for idx, row in df_raw.iterrows():
        text = (str(row['title']) or '') + ' ' + (str(row['content']) or '')
        m = MinHash(num_perm=DEDUP_NUM_PERM)
        for sh in text_shingles(text):
            m.update(sh.encode('utf-8'))
        mh_store[idx] = m
        lsh.insert(str(idx), m)

    # 找 cluster，每個 cluster 保留最早發佈者
    visited = set()
    keep_idx = []
    removed_count = 0
    for idx in range(len(df_raw)):
        if idx in visited:
            continue
        similar = lsh.query(mh_store[idx])
        similar_ids = [int(s) for s in similar]
        if len(similar_ids) > 1:
            # 按時間排序，保留最早
            similar_ids_sorted = sorted(similar_ids, key=lambda i: df_raw.loc[i, 'post_time'])
            keep_idx.append(similar_ids_sorted[0])
            visited.update(similar_ids)
            removed_count += len(similar_ids) - 1
        else:
            keep_idx.append(idx)
            visited.add(idx)

    df_deduped = df_raw.loc[sorted(set(keep_idx))].reset_index(drop=True)
    df_deduped.to_csv(PATH_ARTICLES_DEDUPED, index=False)
    print(f'✅ 去重完成：{len(df_raw)} → {len(df_deduped)} 筆（移除 {removed_count} 筆近重複）')
    print(f'\n去重後平台分布：')
    print(df_deduped['s_name'].value_counts())
else:
    import shutil
    shutil.copy(PATH_ARTICLES_RAW, PATH_ARTICLES_DEDUPED)
    print('⚠️ DEDUP_ENABLED=False，跳過去重')


載入 14571 篇進行去重...
✅ 去重完成：14571 → 13651 筆（移除 942 筆近重複）

去重後平台分布：
s_name
Yahoo股市     8725
Ptt         2393
Yahoo新聞     1601
Dcard        883
Mobile01      49
Name: count, dtype: int64


---
# Step 2 — 貼股價標籤

對每篇文章，取發文當日（或下一個交易日）收盤價，與 D+N_DAYS 交易日後收盤價比較：
- 漲幅 > σ → **+1（看漲）**
- 跌幅 < -σ → **-1（看跌）**
- 否則 → **0（中性，不納入訓練）**

⚠️ **保持 `[-1, 1]` 不轉 `[0, 1]`**。XGBoost/LightGBM/LR 都能處理負值 label，只要不主動呼叫 LabelEncoder。


In [62]:
# ── 取股價 ──
conn = db_connect()
cursor = conn.cursor()
cursor.execute(
    '''
    SELECT trade_date, closing_price
    FROM stock_prices
    WHERE company_id = %s
    ORDER BY trade_date
    ''',
    (COMPANY_ID,)
)
price_rows = cursor.fetchall()
cursor.close()
conn.close()

trade_dates = [row[0] for row in price_rows]
price_dict  = {row[0]: float(row[1]) for row in price_rows}
print(f'共 {len(trade_dates)} 個交易日，範圍：{trade_dates[0]} ～ {trade_dates[-1]}')


共 483 個交易日，範圍：2023-03-01 ～ 2025-02-27


In [63]:
# ── 交易日查詢工具 ──
def get_price_on_or_after(date, trade_dates, price_dict):
    idx = bisect.bisect_left(trade_dates, date)
    if idx >= len(trade_dates):
        return None
    return price_dict[trade_dates[idx]]

def get_nth_trading_day_price(date, n, trade_dates, price_dict):
    idx = bisect.bisect_right(trade_dates, date)
    target_idx = idx + n - 1
    if target_idx >= len(trade_dates):
        return None
    return price_dict[trade_dates[target_idx]]


In [64]:
# ── 貼標 ──
articles = []
with open(PATH_ARTICLES_DEDUPED, newline='', encoding='utf-8') as f:
    articles = list(csv.DictReader(f))
print(f'讀入 {len(articles)} 篇（去重後）')

labeled = []
skip_count = 0

for art in articles:
    try:
        post_date = datetime.strptime(art['post_time'], '%Y-%m-%d %H:%M:%S').date()
    except ValueError:
        try:
            post_date = datetime.strptime(art['post_time'], '%Y-%m-%d').date()
        except Exception:
            skip_count += 1
            continue

    price_d  = get_price_on_or_after(post_date, trade_dates, price_dict)
    price_dn = get_nth_trading_day_price(post_date, N_DAYS, trade_dates, price_dict)

    if price_d is None or price_dn is None or price_d == 0:
        skip_count += 1
        continue

    pct = (price_dn - price_d) / price_d
    if pct > SIGMA:
        label = 1
    elif pct < -SIGMA:
        label = -1
    else:
        label = 0

    art['label']      = label
    art['pct_change'] = round(pct * 100, 2)
    labeled.append(art)

print(f'貼標完成：{len(labeled)} 篇（跳過 {skip_count} 篇）')

fieldnames = ['no', 'post_time', 'title', 'content', 's_name', 'label', 'pct_change']
with open(PATH_ARTICLES_LABELED, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for art in labeled:
        writer.writerow({k: art.get(k, '') for k in fieldnames})

label_cnt = Counter(art['label'] for art in labeled)
print(f'\n看漲 ( 1)：{label_cnt[ 1]} 篇')
print(f'看跌 (-1)：{label_cnt[-1]} 篇')
print(f'中性 ( 0)：{label_cnt[ 0]} 篇（不納入訓練）')


讀入 13651 篇（去重後）
貼標完成：13619 篇（跳過 32 篇）

看漲 ( 1)：5360 篇
看跌 (-1)：4900 篇
中性 ( 0)：3359 篇（不納入訓練）


---
# Step 3 — 斷詞、向量化、切分

## ⭐ 核心變化

**`AGGREGATION_LEVEL`** 決定樣本層級：
- `'article'`（預設）：每篇文章一筆，約 1000+ 筆 → 特徵/樣本比健康
- `'daily'`：同日文章合併成一筆，約 100~200 天 → 樣本少，換股時可快速跑 baseline

**切分流程**（無論哪種 aggregation）：
1. 先切分 train/test 索引
2. **只在 train 上 fit** `TfidfVectorizer` + `SelectKBest(chi2)`
3. 套用到 test（`transform` only）
4. **時序切割時自動加 embargo**：train_end 往前砍 `EMBARGO_DAYS` 曆日


In [41]:
# ── Step 3-1：斷詞（耗時，只跑一次）──
articles = []
with open(PATH_ARTICLES_LABELED, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        row['label'] = int(row['label'])
        articles.append(row)

# 排除中性
articles = [a for a in articles if a['label'] != 0]
print(f'有效文章（排除中性）：{len(articles)} 篇')

print(f'\n斷詞中（method={TOKENIZE_METHOD}）...')
article_tokens_str = []   # 每篇 tokens 字串
article_labels     = []   # 每篇 label（維持 -1/1）
article_dates      = []   # 每篇 post_date
article_sources    = []   # 每篇 source

for i, art in enumerate(articles):
    if i % 200 == 0:
        print(f'  {i}/{len(articles)}')
    post_date = datetime.strptime(art['post_time'], '%Y-%m-%d %H:%M:%S').date()
    text = (art.get('title') or '') + ' ' + (art.get('content') or '')
    tokens = tokenize(text)
    if len(tokens) < 3:  # 斷詞後太短的丟掉
        continue
    article_tokens_str.append(' '.join(tokens))
    article_labels.append(art['label'])
    article_dates.append(post_date)
    article_sources.append(art.get('s_name', ''))

print(f'\n✅ 斷詞完成：{len(article_tokens_str)} 篇')


有效文章（排除中性）：6004 篇

斷詞中（method=monpa）...
  0/6004
  200/6004
  400/6004
  600/6004
  800/6004
  1000/6004
  1200/6004
  1400/6004
  1600/6004
  1800/6004
  2000/6004
  2200/6004
  2400/6004
  2600/6004
  2800/6004
  3000/6004
  3200/6004
  3400/6004
  3600/6004
  3800/6004
 ['FW', 'ORG'] (pos tags) mismatch
  4000/6004
  4200/6004
  4400/6004
 [] (pos tags) mismatch
  4600/6004
  4800/6004
  5000/6004
  5200/6004
  5400/6004
  5600/6004
  5800/6004
  6000/6004

✅ 斷詞完成：6003 篇


In [65]:
# ── Step 3-2：依 AGGREGATION_LEVEL 組裝 corpus ──

if AGGREGATION_LEVEL == 'daily':
    # 日級合併
    day_tokens = defaultdict(list)
    day_labels_dict = {}
    day_sources = defaultdict(list)

    for d, t, l, s in zip(article_dates, article_tokens_str, article_labels, article_sources):
        day_tokens[d].append(t)
        day_labels_dict[d] = l  # 同日多篇 label 相同（D+3 股價一致）
        day_sources[d].append(s)

    sample_dates = sorted(day_tokens.keys())
    corpus  = [' '.join(day_tokens[d]) for d in sample_dates]
    labels  = [day_labels_dict[d]      for d in sample_dates]
    sources = [Counter(day_sources[d]).most_common(1)[0][0] for d in sample_dates]
    print(f'日級合併：{len(corpus)} 天')

elif AGGREGATION_LEVEL == 'article':
    # 維持文章級
    corpus       = article_tokens_str
    labels       = article_labels
    sample_dates = article_dates
    sources      = article_sources
    print(f'文章級保留：{len(corpus)} 篇')

else:
    raise ValueError(f'未知 AGGREGATION_LEVEL: {AGGREGATION_LEVEL}')

print(f'看漲：{labels.count(1)}，看跌：{labels.count(-1)}')

# 儲存 processed
with open(PATH_ARTICLES_PROCESSED, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['date', 'processed_text', 'label', 'source'])
    for d, t, l, s in zip(sample_dates, corpus, labels, sources):
        writer.writerow([d, t, l, s])
print(f'已儲存至 {PATH_ARTICLES_PROCESSED}')


日級合併：271 天
看漲：167，看跌：104
已儲存至 articles_processed.csv


In [66]:
# ── Step 3-3：切分訓練/測試集（含 embargo）──
from sklearn.model_selection import train_test_split as _tts

n = len(corpus)
dates_arr = np.array([pd.Timestamp(d) for d in sample_dates])

if SPLIT_METHOD == 'time':
    # 先按索引切
    order = np.argsort(dates_arr)  # 時序排序
    split_idx = int(n * (1 - TEST_RATIO))
    train_idx_raw = order[:split_idx]
    test_idx_raw  = order[split_idx:]

    # ⭐ Embargo：train 尾端往前砍 EMBARGO_DAYS（Bug D）
    test_start_date = dates_arr[test_idx_raw].min()
    embargo_cutoff  = test_start_date - pd.Timedelta(days=EMBARGO_DAYS)
    train_indices   = train_idx_raw[dates_arr[train_idx_raw] < embargo_cutoff]
    test_indices    = test_idx_raw

    print(f'切割方式：時序（前 {int((1-TEST_RATIO)*100)}% 訓練 / 後 {int(TEST_RATIO*100)}% 測試）')
    print(f'Embargo: train_end < {embargo_cutoff.date()} (砍掉 {len(train_idx_raw) - len(train_indices)} 筆)')
    print(f'Test start: {test_start_date.date()}')

elif SPLIT_METHOD == 'random':
    all_idx = list(range(n))
    train_indices, test_indices = _tts(
        all_idx, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=labels
    )
    train_indices = np.array(train_indices)
    test_indices  = np.array(test_indices)
    print(f'切割方式：隨機 stratified（test_size={TEST_RATIO}, seed={RANDOM_SEED}）')

else:
    raise ValueError(f'未知 SPLIT_METHOD: {SPLIT_METHOD}')

# ── 產出 train/test 變數 ──
corpus_train = [corpus[i] for i in train_indices]
corpus_test  = [corpus[i] for i in test_indices]
y_train      = np.array([labels[i] for i in train_indices])  # ⭐ 維持 [-1, 1]
y_test       = np.array([labels[i] for i in test_indices])
dates_train  = np.array([sample_dates[i] for i in train_indices])
dates_test   = np.array([sample_dates[i] for i in test_indices])

print(f'\n訓練集：{len(corpus_train)} 筆（漲={int(sum(y_train==1))}, 跌={int(sum(y_train==-1))}）')
print(f'測試集：{len(corpus_test)} 筆（漲={int(sum(y_test==1))}, 跌={int(sum(y_test==-1))}）')
print(f'⚠️  y 保持 [-1, 1] 不轉 0/1（Bug A 修正）')


切割方式：隨機 stratified（test_size=0.2, seed=42）

訓練集：216 筆（漲=133, 跌=83）
測試集：55 筆（漲=34, 跌=21）
⚠️  y 保持 [-1, 1] 不轉 0/1（Bug A 修正）


In [67]:
# ── Step 3-4：向量化 + 特徵選取（只在 train fit）──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

if TERM_SELECT_METHOD == 'sklearn':
    # Vectorizer
    vectorizer = TfidfVectorizer(
        use_idf=True, lowercase=False, token_pattern=r'\S+',
        ngram_range=NGRAM_RANGE, sublinear_tf=True, min_df=2
    )
    X_full_train = vectorizer.fit_transform(corpus_train)   # ⭐ 只 fit train
    X_full_test  = vectorizer.transform(corpus_test)
    print(f'原始訓練集：{X_full_train.shape}')

    # Chi2 Selector
    selector = SelectKBest(chi2, k=min(TOP_K_WORDS * 2, X_full_train.shape[1]))
    X_train = selector.fit_transform(X_full_train, y_train)  # ⭐ 只 fit train
    X_test  = selector.transform(X_full_test)
    feature_names = vectorizer.get_feature_names_out()[selector.get_support()]
    print(f'選後訓練集：{X_train.shape}')
    print(f'選後測試集：{X_test.shape}')

    # 顯示選出的特徵
    print(f'\n選出特徵（前 30 個）：')
    print(', '.join(feature_names[:30].tolist()))

    # ── 關鍵詞分析 ──
    bull_mask = (y_train == 1)
    bear_mask = (y_train == -1)
    X_arr = X_train.toarray()
    bull_mean = X_arr[bull_mask].mean(axis=0)
    bear_mean = X_arr[bear_mask].mean(axis=0)
    diff = bull_mean - bear_mean
    order = np.argsort(-np.abs(diff))

    with open(PATH_KEYWORDS_ANALYSIS, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.writer(f)
        w.writerow(['詞', '看漲平均TF-IDF', '看跌平均TF-IDF', '差值', '方向'])
        for i in order:
            w.writerow([
                feature_names[i],
                round(float(bull_mean[i]), 6),
                round(float(bear_mean[i]), 6),
                round(float(diff[i]), 6),
                '看漲' if diff[i] > 0 else '看跌',
            ])
    print(f'\n關鍵詞分析已存 {PATH_KEYWORDS_ANALYSIS}')

else:
    raise NotImplementedError("本次簡化：只保留 'sklearn' 模式。tf/chi2 模式未重構")

# 儲存
sparse.save_npz(PATH_X_TRAIN, X_train)
sparse.save_npz(PATH_X_TEST,  X_test)
np.save(PATH_Y_TRAIN, y_train)
np.save(PATH_Y_TEST,  y_test)
np.save(PATH_DATES_TRAIN, dates_train)
np.save(PATH_DATES_TEST,  dates_test)
sparse.save_npz(PATH_VECTOR_NPZ, X_train)
with open(PATH_FEATURE_NAMES, 'wb') as f:
    pickle.dump(list(feature_names), f)

print(f'\n✅ 所有檔案已儲存')


原始訓練集：(216, 98538)
選後訓練集：(216, 400)
選後測試集：(55, 400)

選出特徵（前 30 個）：
上周 成交量, 上漲 受惠, 上百萬, 下半年 景氣, 下半旬, 下市, 下跌 再度, 下降 美元, 不銹鋼, 中化生, 互見 巴拿, 亞泥, 亞洲 北歐, 交保, 交投 亮點, 交通部 航線, 低點 中國, 俄羅斯 中國, 俄羅斯 原油, 俄羅斯港口, 修船, 修船 活動, 倒數 政策, 值得 觀察, 偉訓, 傳出 航商, 僵持, 內部 稽核, 全球 據點, 全球 貨量

關鍵詞分析已存 keywords_analysis.csv

✅ 所有檔案已儲存


In [68]:
# ── ⭐ Step 3-5：診斷 cell（每次訓練前必跑，檢查 bug）──
print("="*60)
print("診斷報告")
print("="*60)

# 1. y 值域
y_tr_unique = np.unique(y_train, return_counts=True)
y_te_unique = np.unique(y_test, return_counts=True)
print(f"\n[1] y_train unique: {y_tr_unique}")
print(f"    y_test  unique: {y_te_unique}")
y_ok = set(y_tr_unique[0]).issubset({-1, 1}) and set(y_te_unique[0]).issubset({-1, 1})
print(f"    {'✅' if y_ok else '❌'} y 值域應為 [-1, 1]")

# 2. 特徵/樣本比
ratio = X_train.shape[1] / X_train.shape[0]
print(f"\n[2] 訓練集 shape: {X_train.shape}")
print(f"    特徵/樣本比: {ratio:.3f}")
ratio_ok = ratio < 0.5
print(f"    {'✅' if ratio_ok else '⚠️ '} 健康比例應 < 0.5（你的 {ratio:.2f}）")

# 3. 關鍵財經詞
key_words = ['華城', '電力', '訂單', '重電', '輝達', '台達電', '法人', '外資']
found = [w for w in key_words if any(w in f for f in feature_names)]
print(f"\n[3] 關鍵財經詞覆蓋: {len(found)}/{len(key_words)}")
print(f"    有選到: {found}")

# 4. Corpus 長度
corpus_lens = [len(c.split()) for c in corpus_train]
print(f"\n[4] 每筆 tokens 數: min={min(corpus_lens)}, "
      f"median={int(np.median(corpus_lens))}, max={max(corpus_lens)}")

# 5. LogReg sanity check
from sklearn.linear_model import LogisticRegression
_clf = LogisticRegression(class_weight='balanced', max_iter=2000).fit(X_train, y_train)
_pred = _clf.predict(X_test)
pred_dist = np.unique(_pred, return_counts=True)
print(f"\n[5] LogReg 預測分布: {pred_dist}")
pred_ok = len(pred_dist[0]) > 1
print(f"    {'✅' if pred_ok else '⚠️ '} 模型有預測多類")
print("\n" + "="*60)


診斷報告

[1] y_train unique: (array([-1,  1]), array([ 83, 133], dtype=int64))
    y_test  unique: (array([-1,  1]), array([21, 34], dtype=int64))
    ✅ y 值域應為 [-1, 1]

[2] 訓練集 shape: (216, 400)
    特徵/樣本比: 1.852
    ⚠️  健康比例應 < 0.5（你的 1.85）

[3] 關鍵財經詞覆蓋: 2/8
    有選到: ['訂單', '外資']

[4] 每筆 tokens 數: min=162, median=3534, max=13714

[5] LogReg 預測分布: (array([-1,  1]), array([ 9, 46], dtype=int64))
    ✅ 模型有預測多類



---
# Step 4 — 分類器訓練與評估

## ⭐ 修正重點

- **`evaluate` 函式動態取 labels**，不寫死 `[1, -1]`（修 Bug A）
- **Confusion Matrix 改用 sklearn 內建標籤**，避免手動格式化出錯
- **XGBoost 用 `label_encoder` 參數跳過內部轉換**（它會抱怨 `-1` 但仍能訓練）
- **加入 macro-F1**：對不平衡資料更公允
- **⭐ Daily Prediction**：若 `AGGREGATION_LEVEL='article'` 且 `DAILY_PREDICTION=True`，測試時按日 aggregate predict_proba


In [69]:
# ── 載入資料 ──
X_train = sparse.load_npz(PATH_X_TRAIN)
X_test  = sparse.load_npz(PATH_X_TEST)
y_train = np.load(PATH_Y_TRAIN)
y_test  = np.load(PATH_Y_TEST)
dates_train = np.load(PATH_DATES_TRAIN, allow_pickle=True)
dates_test  = np.load(PATH_DATES_TEST,  allow_pickle=True)

print(f'訓練集：{X_train.shape}')
print(f'測試集：{X_test.shape}')
print(f'y_train: {np.unique(y_train, return_counts=True)}')
print(f'y_test : {np.unique(y_test,  return_counts=True)}')


訓練集：(216, 400)
測試集：(55, 400)
y_train: (array([-1,  1]), array([ 83, 133], dtype=int64))
y_test : (array([-1,  1]), array([21, 34], dtype=int64))


In [70]:
# ── ⭐ 評估函式（Bug A 修正版） ──
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, f1_score
)

def daily_aggregate_predict(model, X_te, dates_te, y_te):
    """
    Article-level 訓練 → Daily aggregate 預測。
    每天所有文章的 predict_proba 取平均，argmax 當日級預測。
    """
    if not hasattr(model, 'predict_proba'):
        # 退而求其次：用 decision_function 或 predict 投票
        preds = model.predict(X_te)
        df_ = pd.DataFrame({'date': dates_te, 'pred': preds, 'true': y_te})
        daily = df_.groupby('date').agg(
            pred_sum=('pred', 'sum'),    # 多數決：正數=漲、負數=跌
            true=('true', 'first')       # 同日 true 相同
        )
        daily['pred'] = np.sign(daily['pred_sum']).astype(int)
        daily.loc[daily['pred'] == 0, 'pred'] = 1  # tie → 預設漲
        return daily['pred'].values, daily['true'].values, daily.index.tolist()
    
    proba = model.predict_proba(X_te)
    classes = model.classes_
    df_ = pd.DataFrame(proba, columns=[f'proba_{c}' for c in classes])
    df_['date'] = dates_te
    df_['true'] = y_te
    daily = df_.groupby('date').agg({**{f'proba_{c}': 'mean' for c in classes},
                                     'true': 'first'})
    # argmax over proba columns
    proba_cols = [f'proba_{c}' for c in classes]
    daily['pred'] = daily[proba_cols].values.argmax(axis=1)
    daily['pred'] = daily['pred'].map(lambda i: classes[i])
    return daily['pred'].values, daily['true'].values, daily.index.tolist()


def evaluate(name, model, X_tr, y_tr, X_te, y_te, 
             dates_te=None, daily_predict=False):
    """
    訓練 + 評估。
    動態取 labels，避免 Bug A。
    若 daily_predict=True，會把 article-level 預測按日 aggregate。
    """
    model.fit(X_tr, y_tr)

    # ── ⭐ Daily Prediction 分支 ──
    if daily_predict and dates_te is not None:
        y_pred_daily, y_true_daily, _ = daily_aggregate_predict(model, X_te, dates_te, y_te)
        y_pred, y_te_use = y_pred_daily, y_true_daily
        eval_level = f'Daily ({len(y_true_daily)} 天)'
    else:
        y_pred = model.predict(X_te)
        y_te_use = y_te
        eval_level = f'Article ({len(y_te)} 筆)'

    # ── ⭐ 動態取 labels（Bug A 修正）──
    all_labels = sorted(set(y_te_use) | set(y_pred))
    label_map  = {1: '看漲', -1: '看跌', 0: '中性'}
    target_names = [label_map.get(l, str(l)) for l in all_labels]

    acc = accuracy_score(y_te_use, y_pred)
    f1_macro = f1_score(y_te_use, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_te_use, y_pred, labels=all_labels)

    print(f'\n===== {name} | {eval_level} =====')
    print(f'Accuracy  : {acc*100:.1f}%')
    print(f'Macro-F1  : {f1_macro:.3f}')
    print('\nClassification Report:')
    print(classification_report(y_te_use, y_pred, labels=all_labels,
                                target_names=target_names, zero_division=0))
    print('Confusion Matrix:')
    header = '         ' + '  '.join(f'預測{label_map.get(l, l)}' for l in all_labels)
    print(header)
    for i, true_label in enumerate(all_labels):
        row_name = f'真實{label_map.get(true_label, true_label)}'
        row_vals = '  '.join(f'{v:6d}' for v in cm[i])
        print(f'{row_name:8s}  {row_vals}')

    return {'name': name, 'acc': acc, 'f1_macro': f1_macro, 'cm': cm, 'level': eval_level}


In [71]:
# ── ⭐ 訓練多模型 ──
from sklearn.naive_bayes    import ComplementNB
from sklearn.neighbors      import KNeighborsClassifier
from sklearn.svm            import SVC, LinearSVC
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import RandomForestClassifier, VotingClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.calibration    import CalibratedClassifierCV

import lightgbm as lgb
import xgboost  as xgb

# 判斷是否要 daily aggregate 預測
do_daily = (AGGREGATION_LEVEL == 'article') and DAILY_PREDICTION
if do_daily:
    print(f"⭐ AGGREGATION_LEVEL='article' + DAILY_PREDICTION=True")
    print(f"   → 訓練仍用文章級（{len(y_train)} 筆），預測時按日 aggregate proba")
else:
    print(f"評估層級：{AGGREGATION_LEVEL}")

# ── XGBoost 需要把 y 轉成 [0, 1]（它不吃負值），訓練完再轉回 ──
def make_xgb_wrapper(**kwargs):
    """XGBoost 需要把 -1/1 映射到 0/1（內部），但回傳 proba/predict 都在 [-1, 1] 空間。"""
    from sklearn.base import BaseEstimator, ClassifierMixin
    from sklearn.preprocessing import LabelEncoder as _LE
    class XGBWrapper(BaseEstimator, ClassifierMixin):
        def __init__(self, **params):
            self.params = params
            self.le = _LE()
            self.clf = xgb.XGBClassifier(**params)
            self.classes_ = None
        def fit(self, X, y):
            y_enc = self.le.fit_transform(y)
            self.clf.fit(X, y_enc)
            self.classes_ = self.le.classes_
            return self
        def predict(self, X):
            y_enc = self.clf.predict(X)
            return self.le.inverse_transform(y_enc)
        def predict_proba(self, X):
            return self.clf.predict_proba(X)
    return XGBWrapper(**kwargs)

models = [
    ('Complement NB',       ComplementNB()),
    ('KNN (k=5)',           KNeighborsClassifier(n_neighbors=5)),
    ('Linear SVC (calib.)', CalibratedClassifierCV(
                                LinearSVC(C=0.5, class_weight='balanced', max_iter=3000),
                                cv=3, method='sigmoid')),
    ('Decision Tree',       DecisionTreeClassifier(random_state=RANDOM_SEED, 
                                                    class_weight='balanced',
                                                    min_samples_leaf=5)),
    ('Random Forest',       RandomForestClassifier(n_estimators=500,
                                                    min_samples_leaf=3,
                                                    random_state=RANDOM_SEED,
                                                    class_weight='balanced_subsample',
                                                    n_jobs=-1)),
    ('Logistic Regression', LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000)),
    ('LightGBM',            lgb.LGBMClassifier(
                                n_estimators=500, learning_rate=0.03,
                                num_leaves=31, min_data_in_leaf=20,
                                class_weight='balanced',
                                random_state=RANDOM_SEED, verbose=-1)),
    ('XGBoost',             make_xgb_wrapper(
                                n_estimators=500, learning_rate=0.03,
                                max_depth=5, scale_pos_weight=1.24,
                                random_state=RANDOM_SEED,
                                eval_metric='logloss', verbosity=0)),
]

results = []
for name, model in models:
    try:
        r = evaluate(name, model, X_train, y_train, X_test, y_test,
                     dates_te=dates_test, daily_predict=do_daily)
        results.append(r)
    except Exception as e:
        print(f'\n❌ {name} 失敗: {e}')

# 彙整
print('\n' + '='*60)
print('準確率 + F1 彙整')
print('='*60)
for r in sorted(results, key=lambda x: x['acc'], reverse=True):
    print(f"  {r['name']:22s}: acc={r['acc']*100:5.1f}%  f1={r['f1_macro']:.3f}  [{r['level']}]")


評估層級：daily

===== Complement NB | Article (55 筆) =====
Accuracy  : 67.3%
Macro-F1  : 0.673

Classification Report:
              precision    recall  f1-score   support

          看跌       0.54      0.90      0.68        21
          看漲       0.90      0.53      0.67        34

    accuracy                           0.67        55
   macro avg       0.72      0.72      0.67        55
weighted avg       0.76      0.67      0.67        55

Confusion Matrix:
         預測看跌  預測看漲
真實看跌          19       2
真實看漲          16      18

===== KNN (k=5) | Article (55 筆) =====
Accuracy  : 61.8%
Macro-F1  : 0.382

Classification Report:
              precision    recall  f1-score   support

          看跌       0.00      0.00      0.00        21
          看漲       0.62      1.00      0.76        34

    accuracy                           0.62        55
   macro avg       0.31      0.50      0.38        55
weighted avg       0.38      0.62      0.47        55

Confusion Matrix:
         預測看跌  預測看漲
真實看跌 

In [72]:
# ── ⭐ Soft Voting 集成（延伸實驗）──
voters = [
    ('lr',  LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000)),
    ('svc', CalibratedClassifierCV(
                LinearSVC(C=0.5, class_weight='balanced', max_iter=3000),
                cv=3, method='sigmoid')),
    ('rf',  RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                                   class_weight='balanced_subsample',
                                   n_jobs=-1, random_state=RANDOM_SEED)),
    ('cnb', ComplementNB(alpha=0.3)),
    ('lgb', lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03,
                               num_leaves=31, min_data_in_leaf=20,
                               class_weight='balanced',
                               random_state=RANDOM_SEED, verbose=-1)),
]

vote = VotingClassifier(voters, voting='soft',
                        weights=[1, 1, 1, 0.8, 1.5], n_jobs=-1)

r_vote = evaluate('Soft Voting Ensemble', vote, X_train, y_train, X_test, y_test,
                  dates_te=dates_test, daily_predict=do_daily)
results.append(r_vote)



===== Soft Voting Ensemble | Article (55 筆) =====
Accuracy  : 69.1%
Macro-F1  : 0.662

Classification Report:
              precision    recall  f1-score   support

          看跌       0.61      0.52      0.56        21
          看漲       0.73      0.79      0.76        34

    accuracy                           0.69        55
   macro avg       0.67      0.66      0.66        55
weighted avg       0.68      0.69      0.69        55

Confusion Matrix:
         預測看跌  預測看漲
真實看跌          11      10
真實看漲           7      27


---
# Step 5 — 移動回測（修正版）

## ⭐ 修正重點

- **`WINDOW_TYPE`**：`'trading'`（前 N 個交易日）或 `'calendar'`（前 N 個日曆天）
- **`MIN_TRAIN_SAMPLES`**：訓練集太少直接跳過（避免爛模型出手）
- **per-fold embargo**：每次訓練都 respect `EMBARGO_DAYS`
- **⭐ 使用 sklearn Pipeline**：vectorizer + selector + clf 綁緊，從架構上杜絕 leakage
- **支援 `AGGREGATION_LEVEL` 切換**：daily 用日級 corpus，article 可選日級預測


In [50]:
# ── Step 5-1: 載入處理後資料 ──
# 從 processed CSV 讀回（統一來源，不再重跑斷詞）
df_proc = pd.read_csv(PATH_ARTICLES_PROCESSED, dtype={'label': int})
df_proc['date'] = pd.to_datetime(df_proc['date']).dt.date
df_proc = df_proc.sort_values('date').reset_index(drop=True)

print(f'載入處理後文本：{len(df_proc)} 筆')
print(f'AGGREGATION_LEVEL = {AGGREGATION_LEVEL}')
print(f'日期範圍：{df_proc["date"].min()} ～ {df_proc["date"].max()}')

# 建立日期 → 索引映射（for 交易日窗口）
trade_dates_set = set(trade_dates)   # Step 2 載入的交易日


載入處理後文本：6003 筆
AGGREGATION_LEVEL = article
日期範圍：2023-03-01 ～ 2025-02-21


In [51]:
# ── Step 5-2: Pipeline 工廠函式 ──
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

def make_pipeline(clf, k_features=100):
    """用 sklearn Pipeline 綁 TF-IDF + chi2 + 分類器。
    fit 時自動 per-fold，從架構上保證無 leakage。"""
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=False, token_pattern=r'\S+',
            ngram_range=NGRAM_RANGE, min_df=2, sublinear_tf=True)),
        ('chi2', SelectKBest(chi2, k=k_features)),
        ('clf',  clf),
    ])


def get_train_window(test_date, all_dates, window_type, window_days, embargo_days):
    """根據 WINDOW_TYPE 取訓練窗口的日期範圍。"""
    test_date = pd.Timestamp(test_date).date() if not isinstance(test_date, type(trade_dates[0])) else test_date
    # Embargo：train_end 必須在 test_date - embargo 之前
    embargo_cutoff = test_date - timedelta(days=embargo_days)

    if window_type == 'calendar':
        window_start = test_date - timedelta(days=window_days + embargo_days)
        return window_start, embargo_cutoff

    elif window_type == 'trading':
        # 從 trade_dates 中找到 test_date 之前（embargo 後）的第 N 個交易日
        sorted_trades = sorted(trade_dates_set)
        # bisect_right 找到 embargo_cutoff 之後第一個位置
        idx = bisect.bisect_left(sorted_trades, embargo_cutoff)
        if idx < window_days:
            # 交易日不夠
            return sorted_trades[0], embargo_cutoff
        window_start = sorted_trades[idx - window_days]
        return window_start, embargo_cutoff

    else:
        raise ValueError(f'未知 WINDOW_TYPE: {window_type}')


In [ ]:
# ── Step 5-3: 移動回測主迴圈 ──
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# ── 依 AGGREGATION_LEVEL 決定要 loop 的單位 ──
if AGGREGATION_LEVEL == 'daily':
    # 日級：每天一個 test sample
    test_targets = df_proc.copy()
    print(f'回測單位：每天一筆（共 {len(test_targets)} 天）')
elif AGGREGATION_LEVEL == 'article':
    # 文章級：每篇一個 test sample，最後可選按日 aggregate
    test_targets = df_proc.copy()
    print(f'回測單位：每篇文章（共 {len(test_targets)} 筆）')

results_bt = []

for i, row in test_targets.iterrows():
    test_date = row['date']
    # ── 取訓練窗口 ──
    win_start, win_end = get_train_window(
        test_date, df_proc['date'].values,
        WINDOW_TYPE, WINDOW_DAYS, EMBARGO_DAYS
    )
    train_mask = (df_proc['date'] >= win_start) & (df_proc['date'] < win_end)
    train_df = df_proc[train_mask]

    # ── 門檻檢查 ──
    if len(train_df) < MIN_TRAIN_SAMPLES:
        continue
    if len(train_df['label'].unique()) < 2:
        continue

    # ── 訓練 ──
    train_texts = train_df['processed_text'].tolist()
    train_y     = train_df['label'].values

    # 用 VotingClassifier + Pipeline 包起來
    voters = [
        ('cnb', ComplementNB()),
        ('rf',  RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                        class_weight='balanced_subsample',
                                        n_jobs=-1, random_state=RANDOM_SEED)),
        ('lr',  LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000)),
    ]
    ensemble = VotingClassifier(voters, voting='soft', n_jobs=1)
    pipe = make_pipeline(ensemble, k_features=min(100, len(train_df)))

    try:
        pipe.fit(train_texts, train_y)
        pred_proba = pipe.predict_proba([row['processed_text']])[0]
        classes = pipe.named_steps['clf'].classes_
        pred = classes[pred_proba.argmax()]
        conf = pred_proba.max()
    except Exception as e:
        continue


    VOTE_MARGIN = 0.60  # 在參數區設定

    if conf < VOTE_MARGIN:
        continue  # 信心不夠就不出手，不計入結果

    results_bt.append({
        'date': test_date,
        'pred': int(pred),
        'actual': int(row['label']),
        'confidence': float(conf),
        'n_train': len(train_df),
    })

    if (i+1) % 50 == 0:
        print(f'  進度 {i+1}/{len(test_targets)}')

print(f'\n✅ 回測完成，出手 {len(results_bt)} 次')


回測單位：每篇文章（共 6003 筆）
  進度 250/6003
  進度 300/6003
  進度 350/6003
  進度 400/6003
  進度 450/6003
  進度 500/6003
  進度 550/6003
  進度 600/6003
  進度 650/6003
  進度 700/6003
  進度 750/6003
  進度 800/6003
  進度 850/6003
  進度 900/6003
  進度 950/6003
  進度 1000/6003
  進度 1050/6003
  進度 1150/6003
  進度 1200/6003
  進度 1250/6003
  進度 1300/6003
  進度 1350/6003
  進度 1400/6003
  進度 1450/6003
  進度 1500/6003
  進度 1550/6003
  進度 1600/6003
  進度 1650/6003
  進度 1850/6003
  進度 1900/6003
  進度 1950/6003
  進度 2000/6003
  進度 2050/6003
  進度 2100/6003
  進度 2150/6003
  進度 2200/6003
  進度 2250/6003
  進度 2300/6003
  進度 2350/6003
  進度 2400/6003
  進度 2450/6003
  進度 2500/6003
  進度 2550/6003
  進度 2600/6003
  進度 2650/6003
  進度 2700/6003
  進度 2750/6003
  進度 2800/6003
  進度 2850/6003
  進度 2900/6003
  進度 2950/6003
  進度 3000/6003
  進度 3050/6003
  進度 3100/6003
  進度 3150/6003
  進度 3200/6003
  進度 3250/6003
  進度 3300/6003
  進度 3350/6003
  進度 3400/6003
  進度 3450/6003
  進度 3500/6003
  進度 3550/6003
  進度 3600/6003
  進度 3650/6003
  進度 3700/6003
  進度 

In [53]:
# ── Step 5-4: 回測結果彙整 ──
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report

if len(results_bt) == 0:
    print('❌ 沒有出手，請檢查參數（MIN_TRAIN_SAMPLES 可能太高，或 WINDOW_DAYS 太小）')
else:
    df_bt = pd.DataFrame(results_bt)

    # ── 若 article-level + DAILY_PREDICTION，聚合成日 ──
    if AGGREGATION_LEVEL == 'article' and DAILY_PREDICTION:
        print('⭐ 按日 aggregate predictions...')
        daily_bt = df_bt.groupby('date').agg(
            pred=('pred', lambda x: int(np.sign(x.sum())) or 1),
            actual=('actual', 'first'),
            n_articles=('pred', 'count'),
            avg_conf=('confidence', 'mean'),
        ).reset_index()
        print(f'日級預測：{len(daily_bt)} 天')
        y_pred = daily_bt['pred'].values
        y_true = daily_bt['actual'].values
        level_desc = f'Daily aggregated ({len(daily_bt)} 天)'
    else:
        y_pred = df_bt['pred'].values
        y_true = df_bt['actual'].values
        level_desc = f'{AGGREGATION_LEVEL}-level ({len(df_bt)} 筆)'

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[1, -1])

    print('\n' + '='*60)
    print(f'移動回測結果 | {level_desc}')
    print('='*60)
    print(f'Accuracy : {acc*100:.1f}%')
    print(f'Macro-F1 : {f1_macro:.3f}')
    print('\nConfusion Matrix:')
    print('           預測漲    預測跌')
    print(f'真實漲   {cm[0][0]:7d}  {cm[0][1]:7d}')
    print(f'真實跌   {cm[1][0]:7d}  {cm[1][1]:7d}')
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, labels=[1, -1],
                                target_names=['看漲', '看跌'], zero_division=0))

    # 儲存
    if AGGREGATION_LEVEL == 'article' and DAILY_PREDICTION:
        daily_bt.to_csv('backtest_daily_predictions.csv', index=False)
        print('明細存於 backtest_daily_predictions.csv')
    else:
        df_bt.to_csv('backtest_predictions.csv', index=False)
        print('明細存於 backtest_predictions.csv')


⭐ 按日 aggregate predictions...
日級預測：251 天

移動回測結果 | Daily aggregated (251 天)
Accuracy : 55.4%
Macro-F1 : 0.463

Confusion Matrix:
           預測漲    預測跌
真實漲       121       37
真實跌        75       18

Classification Report:
              precision    recall  f1-score   support

          看漲       0.62      0.77      0.68       158
          看跌       0.33      0.19      0.24        93

    accuracy                           0.55       251
   macro avg       0.47      0.48      0.46       251
weighted avg       0.51      0.55      0.52       251

明細存於 backtest_daily_predictions.csv


In [54]:
# ── Step 5-5: 逐日/逐篇預測明細（前 30 筆）──
if len(results_bt) > 0:
    print(f'{"日期":12s}  預測   實際   結果   信心')
    print('-' * 45)
    target_df = daily_bt if (AGGREGATION_LEVEL == 'article' and DAILY_PREDICTION) else df_bt
    for _, row in target_df.head(30).iterrows():
        pred_s   = '漲' if row['pred']   ==  1 else '跌'
        actual_s = '漲' if row['actual'] ==  1 else '跌'
        ok       = '✓' if row['pred'] == row['actual'] else '✗'
        conf = row.get('confidence', row.get('avg_conf', 0))
        print(f'{str(row["date"]):12s}  {pred_s}      {actual_s}      {ok}    {conf:.3f}')


日期            預測   實際   結果   信心
---------------------------------------------
2023-03-30    漲      漲      ✓    0.570
2023-03-31    漲      漲      ✓    0.585
2023-04-01    漲      漲      ✓    0.548
2023-04-02    漲      漲      ✓    0.535
2023-04-03    跌      漲      ✗    0.530
2023-04-04    跌      漲      ✗    0.594
2023-04-05    漲      漲      ✓    0.561
2023-04-06    漲      漲      ✓    0.575
2023-04-11    漲      漲      ✓    0.572
2023-04-12    漲      漲      ✓    0.585
2023-04-15    漲      跌      ✗    0.571
2023-04-16    漲      跌      ✗    0.572
2023-04-20    漲      跌      ✗    0.572
2023-04-21    漲      跌      ✗    0.564
2023-04-22    漲      跌      ✗    0.572
2023-04-23    漲      跌      ✗    0.553
2023-04-24    漲      跌      ✗    0.572
2023-04-27    漲      跌      ✗    0.531
2023-04-28    漲      跌      ✗    0.535
2023-04-29    漲      跌      ✗    0.526
2023-04-30    漲      跌      ✗    0.531
2023-05-01    跌      跌      ✓    0.550
2023-05-24    跌      漲      ✗    0.521
2023-06-09    跌      漲   

---
# 📊 建議的實驗矩陣

跑完 baseline 之後，建議依下表逐一切換超參數測試：

| 實驗 | AGGREGATION_LEVEL | DAILY_PREDICTION | DEDUP_ENABLED | SPLIT_METHOD | EMBARGO_DAYS |
|------|:---:|:---:|:---:|:---:|:---:|
| **S0 Baseline** | article | False | False | random | 0 |
| **S1 + 時序** | article | False | False | time | 0 |
| **S2 + Embargo** | article | False | False | time | 5 |
| **S3 + 去重** | article | False | True | time | 5 |
| **S4 + 日級投票** | article | **True** | True | time | 5 |
| **S5 換股用** | daily | - | True | time | 5 |

**預期結果**：
- S0 accuracy 虛高（70%+），S1~S4 誠實下降到 55~62%
- 每層 delta 對應一個 leakage 來源 → 這就是你報告的「方法論亮點」

---

# ⏭️ 下一步優化（時間充足時）

1. **NTUSD-Fin 情感詞典**：加 7 維 dense features，hstack 進 TF-IDF
2. **Regime features**：股價 ret_20d, vol_20d, RSI, pct_from_high
3. **Optuna 調 LightGBM**：20 trials，預期 +1~3%
4. **Precision@top-10%**：金融實務更看這個指標

以上在上一輪對話中有完整建議，可直接套用。
